# Granite Embedding Models

In [1]:
import warnings
warnings.filterwarnings('ignore')

## Encode with our model

### Use our models with Sentence Transformers

In [2]:
from sentence_transformers import SentenceTransformer, util

model_path = "ibm-granite/granite-embedding-english-r2"
# Load the Sentence Transformer model
model = SentenceTransformer(model_path)

input_queries = [
    ' Who made the song My achy breaky heart? ',
    'summit define'
    ]

input_passages = [
    "Achy Breaky Heart is a country song written by Don Von Tress. Originally titled Don't Tell My Heart and performed by The Marcy Brothers in 1991. ",
    "Definition of summit for English Language Learners. : 1 the highest point of a mountain : the top of a mountain. : 2 the highest level. : 3 a meeting or series of meetings between the leaders of two or more governments."
    ]

# encode queries and passages. The model produces unnormalized vectors. If your task requires normalized embeddings pass normalize_embeddings=True to encode as below.
query_embeddings = model.encode(input_queries)
passage_embeddings = model.encode(input_passages)

# calculate cosine similarity
print(util.cos_sim(query_embeddings, passage_embeddings))

[2025-09-12 01:09:08,568] [WARNING] [real_accelerator.py:194:get_accelerator] Setting accelerator to CPU. If you have GPU or other accelerator, we were unable to detect it.
[2025-09-12 01:09:08,571] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cpu (auto detect)


/u/awasthyp/ws/setup/envs/mteb/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


### Use our models with Huggingface Transformers

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

model_path = "ibm-granite/granite-embedding-english-r2"

# Load the model and tokenizer
model = AutoModel.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)
model.eval()

input_queries = [
    ' Who made the song My achy breaky heart? ',
    'summit define'
    ]

# tokenize inputs
tokenized_queries = tokenizer(input_queries, padding=True, truncation=True, return_tensors='pt')

# encode queries
with torch.no_grad():
    # Queries
    model_output = model(**tokenized_queries)
    # Perform pooling. granite-embedding-278m-multilingual uses CLS Pooling
    query_embeddings = model_output[0][:, 0]

# normalize the embeddings
query_embeddings = torch.nn.functional.normalize(query_embeddings, dim=1)

## Finetune our model on your own data

In [1]:
from datasets import load_dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.evaluation import TripletEvaluator
from sentence_transformers.losses import CachedMultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers

[2025-09-12 01:18:25,937] [WARNING] [real_accelerator.py:194:get_accelerator] Setting accelerator to CPU. If you have GPU or other accelerator, we were unable to detect it.
[2025-09-12 01:18:25,940] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cpu (auto detect)


/u/awasthyp/ws/setup/envs/mteb/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


In [2]:
lr = 2e-5
model_name = "ibm-granite/granite-embedding-english-r2"
output_dir = f"granite-r2-msmarco-{lr}"

model = SentenceTransformer(model_name)

In [3]:
dataset = load_dataset(
        "sentence-transformers/msmarco-co-condenser-margin-mse-sym-mnrl-mean-v1",
        "triplet-hard",
        split="train",
    )
dataset_dict = dataset.train_test_split(test_size=2_000, seed=43)
train_dataset = dataset_dict["train"].select(range(500))
eval_dataset = dataset_dict["test"]

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/17 [00:00<?, ?it/s]

In [4]:
loss = CachedMultipleNegativesRankingLoss(model, mini_batch_size=512)  # Increase mini_batch_size if you have enough VRAM

args = SentenceTransformerTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=512,
    warmup_ratio=0.05,
    fp16=False,  # Set to False if GPU can't handle FP16
    bf16=True,  # Set to True if GPU supports BF16
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # (Cached)MultipleNegativesRankingLoss benefits from no duplicates
    learning_rate=lr,
    # Optional tracking/debugging parameters:
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
    report_to="none"
)

In [5]:
dev_evaluator = TripletEvaluator(
    anchors=eval_dataset["query"],
    positives=eval_dataset["positive"],
    negatives=eval_dataset["negative"],
    name="msmarco-co-condenser-dev",
    )
dev_evaluator(model)

{'msmarco-co-condenser-dev_cosine_accuracy': 0.9259999990463257}

In [6]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
    evaluator=dev_evaluator,
    )
trainer.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

In [7]:
dev_evaluator(model)

{'msmarco-co-condenser-dev_cosine_accuracy': 0.9259999990463257}

In [ ]:
model.save_pretrained(f"{output_dir}/final")